In [1]:
import polars as pl
from csrio_image2biomass.configs.settings import RAW_DATA_DIR, PROCESSED_DATA_DIR

train = pl.read_csv(RAW_DATA_DIR / "train.csv")
test = pl.read_csv(RAW_DATA_DIR / "test.csv")
train

sample_id,image_path,Sampling_Date,State,Species,Pre_GSHH_NDVI,Height_Ave_cm,target_name,target
str,str,str,str,str,f64,f64,str,f64
"""ID1011485656__Dry_Clover_g""","""train/ID1011485656.jpg""","""2015/9/4""","""Tas""","""Ryegrass_Clover""",0.62,4.6667,"""Dry_Clover_g""",0.0
"""ID1011485656__Dry_Dead_g""","""train/ID1011485656.jpg""","""2015/9/4""","""Tas""","""Ryegrass_Clover""",0.62,4.6667,"""Dry_Dead_g""",31.9984
"""ID1011485656__Dry_Green_g""","""train/ID1011485656.jpg""","""2015/9/4""","""Tas""","""Ryegrass_Clover""",0.62,4.6667,"""Dry_Green_g""",16.2751
"""ID1011485656__Dry_Total_g""","""train/ID1011485656.jpg""","""2015/9/4""","""Tas""","""Ryegrass_Clover""",0.62,4.6667,"""Dry_Total_g""",48.2735
"""ID1011485656__GDM_g""","""train/ID1011485656.jpg""","""2015/9/4""","""Tas""","""Ryegrass_Clover""",0.62,4.6667,"""GDM_g""",16.275
…,…,…,…,…,…,…,…,…
"""ID983582017__Dry_Clover_g""","""train/ID983582017.jpg""","""2015/9/1""","""WA""","""Ryegrass""",0.64,9.0,"""Dry_Clover_g""",0.0
"""ID983582017__Dry_Dead_g""","""train/ID983582017.jpg""","""2015/9/1""","""WA""","""Ryegrass""",0.64,9.0,"""Dry_Dead_g""",0.0
"""ID983582017__Dry_Green_g""","""train/ID983582017.jpg""","""2015/9/1""","""WA""","""Ryegrass""",0.64,9.0,"""Dry_Green_g""",40.94


In [2]:
train = train.select("image_path", "target_name", "target").pivot(index="image_path", on="target_name", values="target").join(train.select("image_path", "State", "Species").unique(), on="image_path")
train

image_path,Dry_Clover_g,Dry_Dead_g,Dry_Green_g,Dry_Total_g,GDM_g,State,Species
str,f64,f64,f64,f64,f64,str,str
"""train/ID498304885.jpg""",0.0,7.4132,22.4868,29.9,22.4868,"""NSW""","""Lucerne"""
"""train/ID786365141.jpg""",0.0,3.3,6.8,10.1,6.8,"""Tas""","""Ryegrass"""
"""train/ID353424190.jpg""",0.0,6.2143,22.7857,29.0,22.7857,"""NSW""","""Fescue"""
"""train/ID802547515.jpg""",0.7525,14.5984,18.9629,34.3138,19.7154,"""Tas""","""Ryegrass_Clover"""
"""train/ID871463897.jpg""",10.0981,19.3547,59.7472,89.2,69.8453,"""NSW""","""Fescue_CrumbWeed"""
…,…,…,…,…,…,…,…
"""train/ID1498398599.jpg""",11.0375,5.6471,1.2834,17.9681,12.321,"""Tas""","""Clover"""
"""train/ID353997899.jpg""",9.5934,35.7574,7.8492,53.2,17.4426,"""Vic""","""Phalaris_Clover"""
"""train/ID1139866256.jpg""",0.0,83.8407,74.0593,157.9,74.0593,"""NSW""","""Fescue"""


In [3]:
from sklearn.model_selection import train_test_split
train, val = train_test_split(
    train, test_size=0.2, random_state=42, shuffle=True, stratify=train[['State', 'Species']]
)

In [5]:
train_cleaned = train.select("image_path", "Dry_Clover_g", "Dry_Dead_g", "Dry_Green_g", "Dry_Total_g", "GDM_g") 
val_cleaned = val.select("image_path", "Dry_Clover_g", "Dry_Dead_g", "Dry_Green_g", "Dry_Total_g", "GDM_g") 
test_df = test.with_columns(pl.lit(0).alias("target"))
test_cleaned = test_df.select("image_path", "target_name", "target").pivot(index="image_path", on="target_name", values="target")
train_cleaned.head(10)

image_path,Dry_Clover_g,Dry_Dead_g,Dry_Green_g,Dry_Total_g,GDM_g
str,f64,f64,f64,f64,f64
"""train/ID896386823.jpg""",2.0794,19.7543,12.9541,34.7878,15.0335
"""train/ID397994621.jpg""",0.0,0.0,81.0,81.0,81.0
"""train/ID1036339023.jpg""",23.0755,2.6135,32.191,57.88,55.2665
"""train/ID344618040.jpg""",0.0,6.7327,14.9081,21.6408,14.9081
"""train/ID1385921939.jpg""",0.0,6.8243,43.6757,50.5,43.6757
"""train/ID1343327476.jpg""",3.1429,3.1429,37.7143,44.0,40.8571
"""train/ID1982662138.jpg""",0.0,1.1594,35.9406,37.1,35.9406
"""train/ID415656958.jpg""",1.0707,11.2424,4.8182,17.1313,5.8889
"""train/ID1183807388.jpg""",0.0,20.9156,20.9156,41.8312,20.9156


In [6]:
test_cleaned

image_path,Dry_Clover_g,Dry_Dead_g,Dry_Green_g,Dry_Total_g,GDM_g
str,i32,i32,i32,i32,i32
"""test/ID1001187975.jpg""",0,0,0,0,0


In [ ]:
# import cv2
# import albumentations as A
# from tqdm.auto import tqdm

# augmented_images_dir = AUGUMENTED_DATA_DIR / "train"
# augmented_images_dir.mkdir(exist_ok=True)

# # Define transformations
# no_transform = A.Compose([])
# h_flip = A.Compose([A.HorizontalFlip(p=1.0)])
# v_flip = A.Compose([A.VerticalFlip(p=1.0)])
# hv_flip = A.Compose([A.HorizontalFlip(p=1.0), A.VerticalFlip(p=1.0)])

# transforms = [
#     (no_transform, ""),
#     (h_flip, "_hflip"),
#     (v_flip, "_vflip"),
#     (hv_flip, "_hvflip")
# ]

# # Create list to store augmented data
# augmented_data = []

# # Process each image in train_cleaned
# for row in tqdm(train_cleaned.iter_rows(named=True), total=len(train_cleaned)):
#     img_path = RAW_DATA_DIR / row['image_path']
#     image = cv2.imread(str(img_path))
    
#     if image is None:
#         continue
    
#     # Get the base filename without extension
#     base_name = row['image_path'].replace('train/', '').replace('.jpg', '')
    
#     for transform, suffix in transforms:
#         # Apply transformation
#         augmented = transform(image=image)['image']
        
#         # Save augmented image
#         aug_filename = f"{base_name}{suffix}.jpg"
#         aug_path = augmented_images_dir / aug_filename
#         cv2.imwrite(str(aug_path), augmented)
        
#         if transform == no_transform:
#             continue
        
#         # Create new row with augmented image path
#         new_row = {
#             'image_path': f"train/{aug_filename}",
#             'Dry_Clover_g': row['Dry_Clover_g'],
#             'Dry_Dead_g': row['Dry_Dead_g'],
#             'Dry_Green_g': row['Dry_Green_g'],
#             'Dry_Total_g': row['Dry_Total_g'],
#             'GDM_g': row['GDM_g']
#         }
#         augmented_data.append(new_row)

# # Create new dataframe with augmented data
# train_augmented = pl.DataFrame(augmented_data)

# # Combine original and augmented data
# train_cleaned_expanded = pl.concat([train_cleaned, train_augmented])

# print(f"Original dataset size: {len(train_cleaned)}")
# print(f"Expanded dataset size: {len(train_cleaned_expanded)}")

  0%|          | 0/357 [00:00<?, ?it/s]

Original dataset size: 357
Expanded dataset size: 1428


In [7]:
train_cleaned.write_csv(PROCESSED_DATA_DIR / "train.csv")
val_cleaned.write_csv(PROCESSED_DATA_DIR / "val.csv")
test_cleaned.write_csv(PROCESSED_DATA_DIR / "test.csv")